# EEG Models Fold Metrics

This notebook loads EEGFormer, EEGNet, EEGNet AP, and EEGNet TR results, computes fold-level accuracy/precision/recall for each model, and reports mean and standard deviation across folds.

In [30]:
from pathlib import Path

import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score


def compute_fold_metrics(subject_df, y_col='true_label', pred_col='subject_pred'):
    rows = []
    for fold, fold_df in subject_df.groupby('fold'):
        y_true = fold_df[y_col].astype(int)
        y_pred = fold_df[pred_col].astype(int)
        rows.append(
            {
                'fold': int(fold),
                'accuracy': accuracy_score(y_true, y_pred),
                'precision': precision_score(y_true, y_pred, zero_division=0),
                'recall': recall_score(y_true, y_pred, zero_division=0),
            }
        )
    return pd.DataFrame(rows).sort_values('fold').reset_index(drop=True)


def load_eegformer_subject_predictions(trial_csv_path):
    trial_df = pd.read_csv(trial_csv_path)
    trial_df['trial_pred'] = (trial_df['prob'] >= 0.5).astype(int)

    subject_df = (
        trial_df.groupby(['fold', 'subject'], as_index=False)
        .agg(
            true_label=('y_true', 'first'),
            positive_vote_fraction=('trial_pred', 'mean'),
        )
        .sort_values(['fold', 'subject'])
        .reset_index(drop=True)
    )
    subject_df['subject_pred'] = (subject_df['positive_vote_fraction'] >= 0.5).astype(int)
    return subject_df[['fold', 'subject', 'true_label', 'subject_pred']]


def load_subject_prediction_table(csv_path):
    df = pd.read_csv(csv_path)
    rename_map = {
        'y_true': 'true_label',
        'majority_pred': 'subject_pred',
        'label': 'true_label',
        'pred': 'subject_pred',
    }
    df = df.rename(columns=rename_map)

    required = {'fold', 'true_label', 'subject_pred'}
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f'Missing required columns in {csv_path}: {sorted(missing)}')

    keep_cols = [c for c in ['fold', 'subject', 'true_label', 'subject_pred'] if c in df.columns]
    return df[keep_cols].copy()

## Fold Metrics For All Models

In [31]:
model_sources = {
    'EEGFormer': {
        'path': Path('../results/eegformer_5fold_run/trial_predictions_5fold.csv'),
        'loader': 'trial',
    },
    'EEGNet': {
        'path': Path('../results/ablation/eegnet.csv'),
        'loader': 'subject',
    },
    'EEGNet AP': {
        'path': Path('../results/ablation/eegnet_ap.csv'),
        'loader': 'subject',
    },
    'EEGNet TR': {
        'path': Path('../results/ablation/eegnet_tr.csv'),
        'loader': 'subject',
    },
}

all_fold_metrics = []

for model_name, cfg in model_sources.items():
    if cfg['loader'] == 'trial':
        subject_preds = load_eegformer_subject_predictions(cfg['path'])
    else:
        subject_preds = load_subject_prediction_table(cfg['path'])

    fold_metrics = compute_fold_metrics(subject_preds)
    fold_metrics.insert(0, 'model', model_name)
    all_fold_metrics.append(fold_metrics)

fold_metrics_all_models = pd.concat(all_fold_metrics, ignore_index=True)

summary_metrics = (
    fold_metrics_all_models.groupby('model')[['accuracy', 'precision', 'recall']]
    .agg(['mean', 'std'])
    .sort_index()
)

display(fold_metrics_all_models.style.format({'accuracy': '{:.1%}', 'precision': '{:.1%}', 'recall': '{:.1%}'}))
display(summary_metrics.style.format('{:.1%}'))

print('Mean and std across folds:')
print(summary_metrics.round(4))

,model,fold,accuracy,precision,recall
0,EEGFormer,0,87.5%,78.6%,100.0%
1,EEGFormer,1,91.7%,86.7%,100.0%
2,EEGFormer,2,91.7%,84.6%,100.0%
3,EEGFormer,3,79.2%,85.7%,80.0%
4,EEGFormer,4,87.5%,78.6%,100.0%
5,EEGNet,0,75.0%,69.2%,81.8%
6,EEGNet,1,91.7%,86.7%,100.0%
7,EEGNet,2,75.0%,64.7%,100.0%
8,EEGNet,3,75.0%,84.6%,73.3%
9,EEGNet,4,87.5%,78.6%,100.0%


Mean and std across folds:
          accuracy         precision          recall        
              mean     std      mean     std    mean     std
model                                                       
EEGFormer   0.8750  0.0510    0.8283  0.0395  0.9600  0.0894
EEGNet      0.8083  0.0812    0.7676  0.0955  0.9103  0.1264
EEGNet AP   0.8167  0.0757    0.7831  0.1067  0.9152  0.1444
EEGNet TR   0.8250  0.0456    0.7875  0.0726  0.9083  0.0960
